In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka \
    great-expectations \
    altair==4.2.2 \
    redis \
    psycopg2-binary

### 환경설정 + vault 연결

In [0]:
import os
import sys
import json
from datetime import datetime
from pyspark.sql import SparkSession

os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()
# vault 연결 후 ADLS OAuth 설정 (Databricks 전용)
vault.get_storage_client("datacopsadls")  # Spark conf에 OAuth 설정

# 설정 확인
spark = SparkSession.getActiveSession()

key = "fs.azure.account.auth.type.datacopsadls.dfs.core.windows.net"
print(f"[INFO] ADLS 인증 방식: {spark.conf.get(key, 'NOT SET')}")
# "OAuth" 가 나와야 정상

In [0]:
# 환경설정 셀 바로 다음에 추가
from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession()

# FileSystem 캐시 초기화
spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.closeAll()
print("[OK] FileSystem 캐시 초기화 완료")

### postgreSQL 이메일 조회 

In [0]:
import requests
import psycopg2
import time

# 도메인별 마지막 알림 발송 시각 (5분 중복 방지)
_last_alert_time   = {}
ALERT_INTERVAL_SEC = 300  # 5분

def get_recipient_emails(domain: str = "", source_name: str = "") -> str:
    """
    v_active_sources VIEW에서 이메일 조회
    source_name(실제 폴더명) 우선, fallback으로 domain 사용
    """
    try:
        conn = psycopg2.connect(
            host=vault.get_secret("db-host"),
            dbname=vault.get_secret("db-name"),
            user=vault.get_secret("db-user"),
            password=vault.get_secret("db-password")
        )
        cur = conn.cursor()

        # source_name 우선 조회 (실제 Bronze 폴더명)
        lookup = source_name if source_name else domain
        cur.execute(
            "SELECT email FROM v_active_sources WHERE bronze_folder = %s",
            (lookup,)
        )
        emails = ";".join([row[0] for row in cur.fetchall()])

        # source_name으로 못 찾으면 domain으로 재시도
        if not emails and source_name and domain and source_name != domain:
            cur.execute(
                "SELECT email FROM v_active_sources WHERE bronze_folder = %s",
                (domain,)
            )
            emails = ";".join([row[0] for row in cur.fetchall()])

        conn.close()
        return emails if emails else vault.get_secret("alert-fallback-email")

    except Exception as e:
        print(f"  [WARN] PostgreSQL 이메일 조회 실패: {e}")
        return vault.get_secret("alert-fallback-email")


def send_quarantine_alert(vault, payload: dict, max_retry: int = 3):
    """Logic App HTTP 트리거로 격리 알림 전송 + notification_logs 기록"""
    url = vault.get_secret("logic-app-quarantine-url")
    status = "failed"

    # notification_settings에서 알림 수신 여부 확인
    try:
        conn = psycopg2.connect(
            host=vault.get_secret("db-host"),
            dbname=vault.get_secret("db-name"),
            user=vault.get_secret("db-user"),
            password=vault.get_secret("db-password")
        )
        cur = conn.cursor()

        # domain_name 기준으로 user_id 조회 후 설정 확인
        cur.execute("""
            SELECT ns.notify_email, ns.quarantine_alert
            FROM notification_settings ns
            JOIN web_users wu ON ns.user_id = wu.user_id
            JOIN data_sources ds ON wu.user_id = ds.user_id
            WHERE ds.bronze_folder = %s
        """, (payload.get("source_name", payload.get("domain", "")),))
        row = cur.fetchone()

        if row:
            notify_email, quarantine_alert = row
            if not notify_email or not quarantine_alert:
                print(f"  [SKIP] 알림 설정 비활성화 (notify_email={notify_email}, quarantine_alert={quarantine_alert})")
                conn.close()
                return
        conn.close()
    except Exception as e:
        print(f"  [WARN] 알림 설정 조회 실패 (기본값으로 진행): {e}")

    # Logic App 호출
    for attempt in range(max_retry):
        try:
            resp = requests.post(url, json=payload, timeout=10)
            if resp.status_code in (200, 202):
                print(f"  [OK] 격리 알림 전송 완료")
                status = "sent"
                break
            else:
                print(f"  [WARN] 알림 전송 실패: {resp.status_code} ({attempt+1}/{max_retry})")
        except Exception as e:
            print(f"  [WARN] 알림 전송 오류 ({attempt+1}/{max_retry}): {e}")
        time.sleep(2 ** attempt)

    if status == "failed":
        print(f"  [ERROR] 격리 알림 최종 전송 실패 (총 {max_retry}회 시도)")

    # notification_logs 기록
    try:
        conn = psycopg2.connect(
            host=vault.get_secret("db-host"),
            dbname=vault.get_secret("db-name"),
            user=vault.get_secret("db-user"),
            password=vault.get_secret("db-password")
        )
        cur = conn.cursor()
        message = (
            f"격리 {payload.get('q_count')}건 | "
            f"critical {payload.get('invalid_count')}건 | "
            f"이상치 {payload.get('anomaly_count')}건 | "
            f"TOP5: {', '.join(payload.get('top_reasons', []))}"
        )
        cur.execute("""
            INSERT INTO notification_logs
                (domain_name, alert_type, channel, message, status)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            payload.get("domain", "unknown"),
            "quarantine",
            payload.get("notify_channel", "email"),
            message,
            status
        ))
        conn.commit()
        conn.close()
        print(f"  [OK] notification_logs 기록 완료 (status={status})")
    except Exception as e:
        print(f"  [WARN] notification_logs 기록 실패: {e}")

### Redis 연결 + 규칙 로드

In [0]:
from redis.cluster import RedisCluster, ClusterNode

redis_host     = vault.get_secret("redis-host")
redis_password = vault.get_secret("redis-password")
redis_port     = int(vault.get_secret("redis-port"))

r = RedisCluster(
    startup_nodes=[ClusterNode(redis_host, redis_port)],
    password=redis_password,
    ssl=True,
    ssl_check_hostname=False,
    decode_responses=True,
    skip_full_coverage_check=True,
    socket_connect_timeout=10,
    socket_timeout=10,
)
r.ping()
print(f"[OK] Redis 연결 완료: {redis_host}:{redis_port}")

ACCOUNT        = "datacopsadls"
BRONZE_BASE    = f"abfss://bronze@{ACCOUNT}.dfs.core.windows.net"
BASE_PATH      = f"abfss://silver@{ACCOUNT}.dfs.core.windows.net"
BASE_PATH_Q    = f"abfss://quarantine@{ACCOUNT}.dfs.core.windows.net"
BASE_PATH_LOGS = f"abfss://logs@{ACCOUNT}.dfs.core.windows.net"

# Bronze 컨테이너에서 도메인 폴더 자동 감지
domains = [
    f.path for f in dbutils.fs.ls(BRONZE_BASE)
    if f.isDir()
    and not f.name.startswith("_")
    and not f.name.startswith("$")
]
print(f"[INFO] 감지된 도메인: {[d.rstrip('/').split('/')[-1] for d in domains]}")

# ── Redis 규칙 키 확인 (참고용) ──────────────────────────
available_keys = r.keys("gx_rules:*")
print(f"[INFO] 사용 가능한 규칙 키: {available_keys}")
# 규칙 생성 및 매칭은 스트리밍 시작 셀에서 도메인별로 자동 처리됨

### Spark 검증 + 이상치 탐지 함수 정의

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()

def validate_spark(df_spark, rules: dict) -> list:
    results = []
    slim_profile = rules.get("slim_profile", {})

    # 모든 검증을 한 번의 agg()로 처리
    agg_exprs = []
    rule_meta = []

    for exp in rules.get("expectations", []):
        col_name   = exp.get("column")
        exp_type   = exp.get("expectation_type")
        kwargs     = exp.get("kwargs", {})
        severity   = exp.get("severity", "warning")

        if col_name not in df_spark.columns:
            continue

        alias = f"fail_{len(agg_exprs)}"

        if exp_type == "expect_column_values_to_not_be_null":
            if "null_when" in slim_profile.get(col_name, {}):
                agg_exprs.append(F.lit(0).alias(alias))
            else:
                agg_exprs.append(
                    F.sum(F.col(col_name).isNull().cast("int")).alias(alias)
                )
        elif exp_type == "expect_column_values_to_be_in_set":
            allowed     = kwargs.get("value_set", [])
            allowed_str = [str(v).lower() for v in allowed]
            if allowed_str:
                agg_exprs.append(
                    F.sum(
                        (F.col(col_name).isNotNull() &
                         ~F.lower(F.col(col_name).cast("string")).isin(allowed_str))
                        .cast("int")
                    ).alias(alias)
                )
            else:
                agg_exprs.append(F.lit(0).alias(alias))
        else:
            agg_exprs.append(F.lit(0).alias(alias))

        rule_meta.append({
            "column": col_name, "rule": exp_type,
            "severity": severity, "alias": alias
        })

    if not agg_exprs:
        return []

    # 단 한 번의 스캔으로 모든 규칙 검증
    agg_result = df_spark.agg(*agg_exprs).collect()[0]

    for meta in rule_meta:
        fail_count = agg_result[meta["alias"]] or 0
        results.append({
            "column":     meta["column"],
            "rule":       meta["rule"],
            "severity":   meta["severity"],
            "passed":     fail_count == 0,
            "fail_count": int(fail_count),
        })

    return results


def detect_anomalies_spark(df_spark, anomaly_rules: list):
    """이상치 탐지 — Spark DataFrame 그대로 처리"""
    df_result = df_spark.withColumn("_anomaly_flags", F.lit(""))

    for rule in anomaly_rules:
        name      = rule.get("name")
        columns   = rule.get("columns", [])
        method    = rule.get("method")
        threshold = rule.get("threshold")
        col       = columns[0] if columns else None

        if not col or col not in df_spark.columns:
            continue
        
        col_type = dict(df_spark.dtypes).get(col, "")
        if method in ("iqr", "zscore") and col_type not in ("int", "bigint", "double", "float", "long"):
            print(f"  [SKIP] {name}: {col} 컬럼이 숫자형이 아님 ({col_type})")
            continue

        try:
            if method == "zscore":
                stats = df_spark.select(
                    F.mean(col).alias("mean"),
                    F.stddev(col).alias("std")
                ).first()
                mean, std = stats["mean"], stats["std"]
                t = threshold or 3
                if std and std > 0:
                    df_result = df_result.withColumn(
                        "_anomaly_flags",
                        F.when(
                            F.abs((F.col(col) - mean) / std) > t,
                            F.concat(F.col("_anomaly_flags"), F.lit(f",{name}"))
                        ).otherwise(F.col("_anomaly_flags"))
                    )

            elif method == "iqr":
                q1, q3 = df_spark.approxQuantile(col, [0.25, 0.75], 0.05)
                iqr = q3 - q1
                t   = threshold or 1.5
                df_result = df_result.withColumn(
                    "_anomaly_flags",
                    F.when(
                        (F.col(col) < q1 - t * iqr) | (F.col(col) > q3 + t * iqr),
                        F.concat(F.col("_anomaly_flags"), F.lit(f",{name}"))
                    ).otherwise(F.col("_anomaly_flags"))
                )

            elif method == "frequency":
                freq  = df_spark.groupBy(col).count()
                stats = freq.select(
                    F.mean("count").alias("mean"),
                    F.stddev("count").alias("std")
                ).first()
                mean, std = stats["mean"], stats["std"]
                t = threshold or 3
                if std and std > 0:
                    high_freq = freq.filter(
                        F.col("count") > mean + t * std
                    ).select(col).rdd.flatMap(lambda x: x).collect()
                    if high_freq:
                        df_result = df_result.withColumn(
                            "_anomaly_flags",
                            F.when(
                                F.col(col).isin(high_freq),
                                F.concat(F.col("_anomaly_flags"), F.lit(f",{name}"))
                            ).otherwise(F.col("_anomaly_flags"))
                        )

        except Exception as e:
            print(f"  [WARN] {name} 탐지 실패: {e}")

    df_silver     = df_result.filter(F.col("_anomaly_flags") == "").drop("_anomaly_flags")
    df_quarantine = df_result.filter(F.col("_anomaly_flags") != "") \
                             .withColumnRenamed("_anomaly_flags", "_quarantine_reason")

    return df_silver, df_quarantine

# ── raw_json 파싱 ─────────────────────────────────────────
def parse_raw_json(df_spark):
    from pyspark.sql.functions import from_json, col
    from pyspark.sql.types import MapType, StringType

    df_parsed = df_spark.withColumn(
        "parsed", from_json(col("raw_json"), MapType(StringType(), StringType()))
    )

    # 전체 배치에서 등장하는 모든 키 합집합으로 추출
    all_keys = (
        df_parsed
        .select(F.explode(F.map_keys("parsed")).alias("key"))
        .distinct()
        .rdd.flatMap(lambda x: x)
        .collect()
    )

    if not all_keys:
        print("  [WARN] raw_json 파싱 실패 - 키 없음")
        return df_spark

    for key in all_keys:
        df_parsed = df_parsed.withColumn(key, col("parsed")[key])

    cols_to_drop = ["parsed", "raw_json", "_source",
                    "kafka_timestamp", "_bronze_loaded_at", "_kafka_topic",
                    "_ingest_ts", "_source_type", "_platform", "_company", "_domain",]
    existing_drops = [c for c in cols_to_drop if c in df_parsed.columns]
    df_parsed = df_parsed.drop(*existing_drops)

    print(f"  [OK] JSON 파싱 완료: {len(all_keys)}개 컬럼 추출")
    return df_parsed


# ── NULL 처리 (Spark 버전) ────────────────────────────────
def apply_null_strategies_spark(df_spark, null_strategies: dict):
    """
    Redis에서 로드한 null_strategies를 Spark DataFrame에 적용
    AI가 생성한 전략대로 null 채우기/제거
    """
    drop_cols = []

    for col_name, strategy in null_strategies.items():
        if col_name not in df_spark.columns:
            continue
        s = strategy.get("strategy")

        if s == "drop":
            drop_cols.append(col_name)
        elif s == "allow":
            pass
        elif s == "fill_default":
            default_val = strategy.get("default_value", "")
            if default_val is not None:
                df_spark = df_spark.fillna({col_name: str(default_val)})
        elif s == "fill_mode":
            mode_row = (
                df_spark.filter(F.col(col_name).isNotNull())
                .groupBy(col_name).count()
                .orderBy(F.desc("count"))
                .limit(1).collect()
            )
            if mode_row:
                df_spark = df_spark.fillna({col_name: str(mode_row[0][col_name])})
        elif s == "fill_mean":
            mean_val = df_spark.select(F.mean(col_name)).collect()[0][0]
            if mean_val is not None:
                df_spark = df_spark.fillna({col_name: mean_val})
        elif s == "fill_median":
            median_val = df_spark.approxQuantile(col_name, [0.5], 0.05)
            if median_val:
                df_spark = df_spark.fillna({col_name: median_val[0]})
        elif s == "fill_forward":
            from pyspark.sql.window import Window
            window = Window.orderBy(F.lit(1)).rowsBetween(
                Window.unboundedPreceding, 0
            )
            df_spark = df_spark.withColumn(
                col_name,
                F.last(col_name, ignorenulls=True).over(window)
            )

    if drop_cols:
        existing_drop = [c for c in drop_cols if c in df_spark.columns]
        if existing_drop:
            before = df_spark.count()
            for c in existing_drop:
                df_spark = df_spark.filter(F.col(c).isNotNull())
            after = df_spark.count()
            print(f"  [drop] {before - after}행 제거 | 컬럼: {existing_drop}")

    return df_spark

print("[OK] Spark 검증 함수 정의 완료")

### 텍스트 품질 검사 함수

In [0]:
import re
from openai import AzureOpenAI

# ── 텍스트 품질 검사 ──────────────────────────────────────

# 1차 규칙 기반 패턴
_PROFANITY_KEYWORDS = [
    "fuck", "shit", "asshole", "bitch", "bastard", "damn", "crap",
    "씨발", "개새끼", "병신", "지랄", "좆", "꺼져", "닥쳐"
]
_HATE_KEYWORDS = [
    "nigger", "faggot", "retard", "chink", "spic", "wetback",
    "홍어", "짱깨", "쪽바리", "된장녀", "한남충", "김치녀"
]
_PII_PATTERNS = [
    r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+",  # 이메일
    r"\b\d{3}[-.\s]?\d{3,4}[-.\s]?\d{4}\b",               # 전화번호
    r"\b\d{6}[-]\d{7}\b",                                   # 주민번호
    r"\b(?:\d{4}[-\s]?){4}\b",                              # 신용카드
]
_SPAM_PATTERNS = [
    r"(https?://\S+){3,}",          # URL 3개 이상
    r"(.)\1{5,}",                    # 같은 문자 5번 이상 반복
    r"[!?]{4,}",                     # 특수문자 4개 이상 연속
    r"(\b\w+\b)(\s+\1){3,}",        # 같은 단어 3번 이상 반복
]

def _rule_based_score(text: str, checks: list,
                      keywords: dict = None, patterns: dict = None) -> tuple:
    """
    1차 규칙 기반 점수 계산
    keywords/patterns: 규칙에서 가져온 도메인 특화 기준 (없으면 기본값 사용)
    """
    if not text or not isinstance(text, str):
        return 0.0, []

    text_lower = text.lower()
    matched    = []

    kw_profanity  = (keywords or {}).get("profanity",  _PROFANITY_KEYWORDS)
    kw_hate       = (keywords or {}).get("hate_speech", _HATE_KEYWORDS)
    pt_pii        = (patterns or {}).get("pii",         _PII_PATTERNS)
    pt_spam       = (patterns or {}).get("spam",        _SPAM_PATTERNS)

    if "profanity" in checks:
        if any(kw in text_lower for kw in kw_profanity):
            matched.append("profanity")

    if "hate_speech" in checks:
        if any(kw in text_lower for kw in kw_hate):
            matched.append("hate_speech")

    if "pii" in checks:
        if any(re.search(p, text) for p in pt_pii):
            matched.append("pii")

    if "spam" in checks:
        if any(re.search(p, text_lower) for p in pt_spam):
            matched.append("spam")

    score = min(len(matched) * 0.4 + (0.5 if matched else 0.0), 1.0) if matched else 0.0
    return score, matched


def _gpt_check(texts_with_idx: list, checks: list) -> dict:
    """
    2차 GPT 검사 — 애매한 행만 전달
    반환: {idx: {"flagged": bool, "reasons": list}}
    """
    if not texts_with_idx:
        return {}

    try:
        gpt_client = AzureOpenAI(
            api_key=vault.get_secret("gx-rulegen-openai-key"),
            azure_endpoint=vault.get_secret("gx-rulegen-openai-endpoint"),
            api_version="2024-12-01-preview",
        )
        deployment = vault.get_secret("gx-rulegen-deployment-gpt-4-1-mini")

        items_str = "\n".join(
            f'{i}. "{text}"' for i, (idx, text) in enumerate(texts_with_idx)
        )
        checks_str = ", ".join(checks)

        prompt = f"""다음 텍스트들을 검사하세요. 검사 항목: {checks_str}
각 항목에 대해 문제가 있으면 flagged=true, 없으면 false로 판단하세요.

텍스트 목록:
{items_str}

JSON 형식으로만 응답하세요:
{{
  "results": [
    {{"index": 0, "flagged": true/false, "reasons": ["profanity", "spam", ...]}},
    ...
  ]
}}"""

        response = gpt_client.chat.completions.create(
            model=deployment,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=1000,
        )
        raw = response.choices[0].message.content.strip()
        parsed = json.loads(raw)

        result = {}
        for item in parsed.get("results", []):
            original_idx = texts_with_idx[item["index"]][0]
            result[original_idx] = {
                "flagged":  item.get("flagged", False),
                "reasons":  item.get("reasons", []),
            }
        return result

    except Exception as e:
        print(f"  [WARN] GPT 텍스트 품질 검사 실패: {e}")
        return {}


def apply_text_quality_checks(df_silver: "DataFrame", text_quality_columns: list) -> tuple:
    """
    텍스트 품질 검사 통합 함수 (Spark UDF 기반 — toPandas 제거)
    1차: Spark UDF 규칙 기반 (score >= 0.8 → 즉시 격리, 0.3~0.8 → AMBIGUOUS)
    2차: AMBIGUOUS 행만 GPT 판정 (소량이므로 드라이버 collect 안전)
    반환: (df_clean, df_text_quarantine)
    """
    from pyspark.sql.types import StringType
    from pyspark.sql.functions import udf

    if not text_quality_columns:
        return df_silver, None

    # ── _row_id 먼저 부착 (join 기준키) ──────────────────
    df_result = df_silver.withColumn("_row_id", F.monotonically_increasing_id()) \
                         .withColumn("_text_quarantine_reason", F.lit(""))

    for col_info in text_quality_columns:
        col_name      = col_info.get("column")
        checks        = col_info.get("checks", [])
        keywords      = col_info.get("keywords", {})
        patterns      = col_info.get("patterns", {})

        if col_name not in df_silver.columns:
            continue

        checks_json   = json.dumps(checks)
        keywords_json = json.dumps(keywords)
        patterns_json = json.dumps(patterns)

        # ── 1차: Spark UDF 규칙 기반 ──────────────────────
        @udf(returnType=StringType())
        def text_quality_udf(text, c_json, kw_json, pt_json):
            import json as _json
            checks_   = _json.loads(c_json)
            keywords_ = _json.loads(kw_json)
            patterns_ = _json.loads(pt_json)
            score, matched = _rule_based_score(
                str(text) if text else "", checks_, keywords_, patterns_
            )
            if score >= 0.8:
                return ",".join(f"ERROR_TEXT_{m.upper()}" for m in matched)
            elif score >= 0.3:
                return "AMBIGUOUS"
            return None

        df_result = df_result.withColumn(
            "_text_quarantine_reason",
            F.when(
                F.col("_text_quarantine_reason") == "",
                text_quality_udf(
                    F.col(col_name),
                    F.lit(checks_json),
                    F.lit(keywords_json),
                    F.lit(patterns_json)
                )
            ).otherwise(F.col("_text_quarantine_reason"))
        )

    # ── 2차: AMBIGUOUS 행만 GPT 검사 ─────────────────────
    # df_result에서 파생해야 _row_id가 동일하게 유지됨
    df_ambiguous  = df_result.filter(F.col("_text_quarantine_reason") == "AMBIGUOUS")
    ambiguous_count = df_ambiguous.count()

    if ambiguous_count > 0:
        print(f"  [텍스트] GPT 2차 검사 대상: {ambiguous_count}행")

        all_checks = list({
            c
            for col_info in text_quality_columns
            for c in col_info.get("checks", [])
        })
        text_cols = [
            col_info.get("column")
            for col_info in text_quality_columns
            if col_info.get("column") in df_silver.columns
        ]

        texts_with_idx = [
            (
                row["_row_id"],
                " ".join(str(row[c] or "") for c in text_cols)
            )
            for row in df_ambiguous.select("_row_id", *text_cols).collect()
        ]

        gpt_results = _gpt_check(texts_with_idx, all_checks)

        gpt_rows = [
            (
                idx,
                ",".join(f"ERROR_TEXT_{r.upper()}" for r in res["reasons"])
                if res["flagged"] else "PASS"
            )
            for idx, res in gpt_results.items()
        ]

        if gpt_rows:
            gpt_df = spark.createDataFrame(gpt_rows, ["_row_id", "_gpt_result"])
            df_result = (
                df_result
                .join(gpt_df, on="_row_id", how="left")
                .withColumn(
                    "_text_quarantine_reason",
                    F.when(
                        F.col("_text_quarantine_reason") == "AMBIGUOUS",
                        F.coalesce(F.col("_gpt_result"), F.lit("PASS"))
                    ).otherwise(F.col("_text_quarantine_reason"))
                )
                .drop("_gpt_result")
            )

    # ── 분리: 격리 / 정상 ─────────────────────────────────
    _bad = (
        F.col("_text_quarantine_reason").isNotNull() &
        ~F.col("_text_quarantine_reason").isin("", "PASS", "AMBIGUOUS")
    )

    df_text_quarantine = (
        df_result
        .filter(_bad)
        .withColumnRenamed("_text_quarantine_reason", "_quarantine_reason")
        .drop("_row_id")
    )

    df_clean = (
        df_result
        .filter(~_bad)
        .drop("_text_quarantine_reason", "_row_id")
    )

    text_q_count = df_text_quarantine.count()
    if text_q_count > 0:
        print(f"  [텍스트] 격리: {text_q_count}행")
    else:
        print(f"  [텍스트] 품질 문제 없음")

    return df_clean, (df_text_quarantine if text_q_count > 0 else None)

### 텍스트 품질 검사 함수

In [0]:
import re
from openai import AzureOpenAI

# ── 텍스트 품질 검사 ──────────────────────────────────────

# 1차 규칙 기반 패턴
_PROFANITY_KEYWORDS = [
    "fuck", "shit", "asshole", "bitch", "bastard", "damn", "crap",
    "씨발", "개새끼", "병신", "지랄", "좆", "꺼져", "닥쳐"
]
_HATE_KEYWORDS = [
    "nigger", "faggot", "retard", "chink", "spic", "wetback",
    "홍어", "짱깨", "쪽바리", "된장녀", "한남충", "김치녀"
]
_PII_PATTERNS = [
    r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+",  # 이메일
    r"\b\d{3}[-.\s]?\d{3,4}[-.\s]?\d{4}\b",               # 전화번호
    r"\b\d{6}[-]\d{7}\b",                                   # 주민번호
    r"\b(?:\d{4}[-\s]?){4}\b",                              # 신용카드
]
_SPAM_PATTERNS = [
    r"(https?://\S+){3,}",          # URL 3개 이상
    r"(.)\1{5,}",                    # 같은 문자 5번 이상 반복
    r"[!?]{4,}",                     # 특수문자 4개 이상 연속
    r"(\b\w+\b)(\s+\1){3,}",        # 같은 단어 3번 이상 반복
]

def _rule_based_score(text: str, checks: list,
                      keywords: dict = None, patterns: dict = None) -> tuple:
    """
    1차 규칙 기반 점수 계산
    keywords/patterns: 규칙에서 가져온 도메인 특화 기준 (없으면 기본값 사용)
    """
    if not text or not isinstance(text, str):
        return 0.0, []

    text_lower = text.lower()
    matched    = []

    kw_profanity  = (keywords or {}).get("profanity",  _PROFANITY_KEYWORDS)
    kw_hate       = (keywords or {}).get("hate_speech", _HATE_KEYWORDS)
    pt_pii        = (patterns or {}).get("pii",         _PII_PATTERNS)
    pt_spam       = (patterns or {}).get("spam",        _SPAM_PATTERNS)

    if "profanity" in checks:
        if any(kw in text_lower for kw in kw_profanity):
            matched.append("profanity")

    if "hate_speech" in checks:
        if any(kw in text_lower for kw in kw_hate):
            matched.append("hate_speech")

    if "pii" in checks:
        if any(re.search(p, text) for p in pt_pii):
            matched.append("pii")

    if "spam" in checks:
        if any(re.search(p, text_lower) for p in pt_spam):
            matched.append("spam")

    score = min(len(matched) * 0.4 + (0.5 if matched else 0.0), 1.0) if matched else 0.0
    return score, matched


def _gpt_check(texts_with_idx: list, checks: list) -> dict:
    """
    2차 GPT 검사 — 애매한 행만 전달
    반환: {idx: {"flagged": bool, "reasons": list}}
    """
    if not texts_with_idx:
        return {}

    try:
        gpt_client = AzureOpenAI(
            api_key=vault.get_secret("gx-rulegen-openai-key"),
            azure_endpoint=vault.get_secret("gx-rulegen-openai-endpoint"),
            api_version="2024-12-01-preview",
        )
        deployment = vault.get_secret("gx-rulegen-deployment-gpt-4-1-mini")

        items_str = "\n".join(
            f'{i}. "{text}"' for i, (idx, text) in enumerate(texts_with_idx)
        )
        checks_str = ", ".join(checks)

        prompt = f"""다음 텍스트들을 검사하세요. 검사 항목: {checks_str}
각 항목에 대해 문제가 있으면 flagged=true, 없으면 false로 판단하세요.

텍스트 목록:
{items_str}

JSON 형식으로만 응답하세요:
{{
  "results": [
    {{"index": 0, "flagged": true/false, "reasons": ["profanity", "spam", ...]}},
    ...
  ]
}}"""

        response = gpt_client.chat.completions.create(
            model=deployment,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=1000,
        )
        raw = response.choices[0].message.content.strip()
        parsed = json.loads(raw)

        result = {}
        for item in parsed.get("results", []):
            original_idx = texts_with_idx[item["index"]][0]
            result[original_idx] = {
                "flagged":  item.get("flagged", False),
                "reasons":  item.get("reasons", []),
            }
        return result

    except Exception as e:
        print(f"  [WARN] GPT 텍스트 품질 검사 실패: {e}")
        return {}


def apply_text_quality_checks(df_silver: "DataFrame", text_quality_columns: list) -> tuple:
    """
    텍스트 품질 검사 통합 함수
    1차: 규칙 기반 (score >= 0.8 → 즉시 격리)
    2차: GPT (0.3 <= score < 0.8 → GPT 판정)
    반환: (df_clean, df_text_quarantine)
    """
    if not text_quality_columns:
        return df_silver, None

    # Pandas로 변환해서 처리 (텍스트 검사는 행 단위)
    pdf = df_silver.toPandas()
    quarantine_indices = set()
    quarantine_reasons = {}

    for col_info in text_quality_columns:
        col_name = col_info.get("column")
        checks   = col_info.get("checks", [])

        if col_name not in pdf.columns:
            continue

        ambiguous = []  # GPT 2차 검사 대상

        keywords = col_info.get("keywords", {})
        patterns = col_info.get("patterns", {})

        for idx, text in pdf[col_name].items():
            score, matched = _rule_based_score(
                str(text) if text else "", checks, keywords, patterns
            )

            if score >= 0.8:
                # 확실히 문제 → 즉시 격리
                quarantine_indices.add(idx)
                existing = quarantine_reasons.get(idx, "")
                quarantine_reasons[idx] = existing + "," + ",".join(
                    f"ERROR_TEXT_{m.upper()}" for m in matched
                )
            elif score >= 0.3:
                # 애매함 → GPT 2차 검사 대상
                ambiguous.append((idx, str(text)))

        # 2차 GPT 검사
        if ambiguous:
            print(f"  [텍스트] {col_name}: GPT 2차 검사 {len(ambiguous)}행")
            gpt_results = _gpt_check(ambiguous, checks)
            for idx, result in gpt_results.items():
                if result["flagged"]:
                    quarantine_indices.add(idx)
                    existing = quarantine_reasons.get(idx, "")
                    quarantine_reasons[idx] = existing + "," + ",".join(
                        f"ERROR_TEXT_{r.upper()}" for r in result["reasons"]
                    )

    if not quarantine_indices:
        print(f"  [텍스트] 품질 문제 없음")
        return df_silver, None

    # quarantine_reason 컬럼 추가
    pdf["_quarantine_reason"] = pdf.index.map(
        lambda i: quarantine_reasons.get(i, "")
    )

    df_text_quarantine = spark.createDataFrame(
        pdf[pdf.index.isin(quarantine_indices)]
    )
    df_clean = spark.createDataFrame(
        pdf[~pdf.index.isin(quarantine_indices)].drop(columns=["_quarantine_reason"])
    )

    print(f"  [텍스트] 격리: {len(quarantine_indices)}행")
    return df_clean, df_text_quarantine

### process_batch 정의 + 스트리밍 시작

In [0]:
def process_batch(df_spark, epoch_id, source_name="unknown"):
    """
    Structured Streaming foreachBatch 콜백
    흐름: 파싱 → persist → 검증+태깅 → critical 위반 격리 → 이상치 탐지 → 텍스트 품질 검사 → 저장 → 로그
    NULL 처리(apply_null_strategies)는 Silver→Gold 레이어에서 수행
    """
    if not df_spark.take(1):
        return

    print(f"\n[BATCH {epoch_id}][{source_name}] 배치 수신")

    # 1. raw_json 파싱
    df_parsed = parse_raw_json(df_spark)

    # 2. source_name 기반으로 Redis 규칙 로드
    matched_key = None
    for key in r.keys("gx_rules:*"):
    # Redis 키 자체가 source_name 기반이므로 키 이름으로 직접 비교
        key_source = key.replace("gx_rules:", "")
        if key_source == source_name.lower():
            matched_key = key
            break

    if matched_key is None:
        print(f"[BATCH {epoch_id}][{source_name}] 규칙 없음 → 스킵")
        df_parsed.unpersist()
        return

    current_rules = json.loads(r.get(matched_key))

    # ── persist ①: df_parsed 고정
    #    이후 validate_spark() + 태깅 루프가 재스캔 없이 메모리에서 수행됨 ──
    from pyspark import StorageLevel
    df_parsed.persist(StorageLevel.MEMORY_AND_DISK)

    # finally에서 NameError 방지용 초기화
    df_flagged          = None
    df_silver           = None
    df_anomaly          = None
    df_text_quarantine  = None
    q_path = None


    try:
        # 3. 검증 — validate_spark()는 단일 agg()로 전체 규칙 처리 (1회 스캔)
        results      = validate_spark(df_parsed, current_rules)
        total        = len(results)
        passed_count = sum(1 for res in results if res["passed"])

        # 4. critical 위반 태깅 (persist된 df_parsed 재사용, 추가 스캔 없음)
        df_flagged = df_parsed.withColumn("_error_codes", F.lit(""))

        for exp in current_rules.get("expectations", []):
            col_name   = exp.get("column")
            exp_type   = exp.get("expectation_type")
            kwargs     = exp.get("kwargs", {})
            error_code = exp.get("error_code", f"ERROR_{col_name}_INVALID".upper())
            severity   = exp.get("severity", "warning")

            if col_name not in df_parsed.columns:
                continue
            if severity != "critical":
                continue

            try:
                if exp_type == "expect_column_values_to_not_be_null":
                    df_flagged = df_flagged.withColumn(
                        "_error_codes",
                        F.when(
                            F.col(col_name).isNull(),
                            F.concat(F.col("_error_codes"), F.lit(f",{error_code}"))
                        ).otherwise(F.col("_error_codes"))
                    )
                elif exp_type == "expect_column_values_to_be_in_set":
                    allowed     = kwargs.get("value_set", [])
                    allowed_str = [str(v).lower() for v in allowed]
                    if allowed_str:
                        df_flagged = df_flagged.withColumn(
                            "_error_codes",
                            F.when(
                                F.col(col_name).isNotNull() &
                                ~F.lower(F.col(col_name).cast("string")).isin(allowed_str),
                                F.concat(F.col("_error_codes"), F.lit(f",{error_code}"))
                            ).otherwise(F.col("_error_codes"))
                        )
            except Exception as e:
                print(f"  [WARN] {col_name} 태깅 실패: {e}")

        # ── persist ②: df_flagged 고정
        #    valid/invalid 분기 + count() 2회 시 재스캔 방지 ──
        df_flagged.persist(StorageLevel.MEMORY_AND_DISK)

        # 5. valid / invalid 분리
        df_valid   = df_flagged.filter(F.col("_error_codes") == "").drop("_error_codes")
        df_invalid = df_flagged.filter(F.col("_error_codes") != "") \
                               .withColumnRenamed("_error_codes", "_quarantine_reason")

        # ── count()는 persist 이후에만 호출
        #    invalid_count 1번 → valid_count는 계산으로 대체 (count() 절약) ──
        invalid_count = df_invalid.count()
        valid_count   = df_flagged.count() - invalid_count
        print(f"  [검증] 정상: {valid_count}행 | critical 위반: {invalid_count}행")

        # 6. 이상치 탐지
        df_silver, df_anomaly = detect_anomalies_spark(
            df_valid, current_rules.get("anomaly_rules", [])
        )

        # ── persist ③④: silver/anomaly 고정 (write 시 재스캔 방지) ──
        df_silver.persist(StorageLevel.MEMORY_AND_DISK)
        df_anomaly.persist(StorageLevel.MEMORY_AND_DISK)

        silver_count  = df_silver.count()
        anomaly_count = df_anomaly.count()
        print(f"  [이상치] 정상: {silver_count}행 | 이상치: {anomaly_count}행")

        # ── 6-1. 텍스트 품질 검사 ──────────────────────────
        #    1차: 규칙 기반 (score >= 0.8 → 즉시 격리)
        #    2차: GPT (0.3 <= score < 0.8 → GPT 판정)
        #    문제 있는 행은 Quarantine으로 격리 ──
        text_quality_cols = current_rules.get("text_quality_columns", [])
        df_silver, df_text_quarantine = apply_text_quality_checks(
            df_silver, text_quality_cols
        )
        text_quarantine_count = df_text_quarantine.count() if df_text_quarantine is not None else 0
        silver_count = silver_count - text_quarantine_count
        print(f"  [텍스트] Silver 최종: {silver_count}행 | 텍스트 격리: {text_quarantine_count}행")

        # 7. Quarantine 합치기 (critical 위반 + 이상치 + 텍스트 품질)
        q_count = anomaly_count + invalid_count + text_quarantine_count
        dfs_to_union = [
            df for df in [df_invalid, df_anomaly, df_text_quarantine]
            if df is not None
        ]
        if dfs_to_union:
            df_quarantine = dfs_to_union[0]
            for df in dfs_to_union[1:]:
                df_quarantine = df_quarantine.union(df)
        else:
            df_quarantine = None

        # 8. 메타 추가
        processed_at = datetime.utcnow().isoformat() + "Z"
        run_ts       = datetime.utcnow().strftime("%H%M%S")
        today        = datetime.utcnow().strftime("%Y-%m-%d")
        domain       = current_rules.get("domain", "unknown_domain")
        display_domain = domain  # 메일 본문용 (AI 생성 도메인명)


        # ── Silver: 검사용 내부 컬럼(_로 시작) 제거 후 순수 비즈니스 컬럼만 적재 ──
        internal_cols   = [c for c in df_silver.columns if c.startswith("_")]
        df_silver_clean = df_silver.drop(*internal_cols)

        if q_count > 0 and df_quarantine is not None:
            df_quarantine = (
                df_quarantine
                .withColumn("_quarantine_ts", F.lit(processed_at))
                .withColumn("_processed_at",  F.lit(processed_at))
                .withColumn("_run_id",        F.lit(run_ts))
                .withColumn("_epoch_id",      F.lit(epoch_id))
                .withColumn(                                          # ← 여기 추가
                    "_row_hash",
                    F.md5(F.concat_ws(",", *[
                        c for c in df_quarantine.columns
                        if not c.startswith("_")
                    ]))
                )
            )
            q_path = f"{BASE_PATH_Q}/{domain}/date={today}/epoch={epoch_id}"
            df_quarantine.write.mode("overwrite").parquet(q_path)

        # ── 9. Silver 저장
        #    epoch_id 사용: 배치마다 고유 경로 보장, 병렬 쓰기 충돌 방지 ──
        silver_path = f"{BASE_PATH}/{domain}/date={today}/epoch={epoch_id}"
        df_silver_clean.write.mode("overwrite").parquet(silver_path)

        # ── 10. Quarantine 저장 + Logic App 알림 ──
        if q_count > 0 and df_quarantine is not None:

            # ── 10-1. 격리 사유 집계
            #    저장된 parquet에서 읽어 재스캔 안전하게 처리 ──
            reason_rows = (
                spark.read.parquet(q_path)
                .groupBy("_quarantine_reason")
                .count()
                .orderBy(F.desc("count"))
                .limit(5)
                .collect()
            )
            top_reasons = [
                f"{row['_quarantine_reason']} ({row['count']}건)"
                for row in reason_rows
            ]

            # ── 10-2. Logic App 알림 호출 (5분 간격 제한) ──
            now  = time.time()
            last = _last_alert_time.get(source_name, 0)

            if now - last >= ALERT_INTERVAL_SEC:
                send_quarantine_alert(vault, {
                    "domain":                domain,
                    "epoch_id":              epoch_id,
                    "q_count":               q_count,
                    "invalid_count":         invalid_count,
                    "anomaly_count":         anomaly_count,
                    "text_quarantine_count": text_quarantine_count,
                    "top_reasons":           top_reasons,
                    "processed_at":          processed_at,
                    "q_path":                q_path,
                    "recipient_emails": get_recipient_emails(domain=domain, source_name=source_name),
                    "notify_channel":        "email",
                })
                _last_alert_time[source_name] = now
            else:
                print(f"  [SKIP] 알림 생략 ({int(now - last)}초 전 발송됨, {ALERT_INTERVAL_SEC}초 간격)")

        weighted_deduction = (
            invalid_count * 1.0 +
            anomaly_count * 0.5 +
            text_quarantine_count * 0.3
            )
        quality = round(max(0, 100 - (weighted_deduction / total * 100)), 1) if total > 0 else 0
        print(f"[BATCH {epoch_id}] 완료 | Silver {silver_count}행 | Quarantine {q_count}행 | 품질 {quality}%")

        # 11. GX 로그 저장
        gx_log = {
            "run_id":                processed_at,
            "epoch_id":              epoch_id,
            "source":                domain,
            "date":                  today,
            "total_rules":           total,
            "passed":                passed_count,
            "failed":                total - passed_count,
            "quality_score":         quality,
            "silver_rows":           silver_count,
            "quarantine_rows":       q_count,
            "text_quarantine_rows":  text_quarantine_count,
        }
        log_path = f"{BASE_PATH_LOGS}/gx_runs/date={today}/run_{processed_at.replace(':', '-')}.json"
        log_json = json.dumps(gx_log, ensure_ascii=False, indent=2)
        spark.createDataFrame([(log_json,)], ["log"]) \
             .write.mode("overwrite").text(log_path)
        print(f"  [OK] GX 로그 저장: {log_path}")

    finally:
        # ── 반드시 unpersist: 배치가 끝나면 메모리 해제
        #    None 체크로 NameError 방지 (태깅/탐지 도중 예외 시에도 안전) ──
        df_parsed.unpersist()
        if df_flagged is not None:
            df_flagged.unpersist()
        if df_silver is not None:
            df_silver.unpersist()
        if df_anomaly is not None:
            df_anomaly.unpersist()
        if df_text_quarantine is not None:
            df_text_quarantine.unpersist()

### 도메인별 스트리밍 시작 공통 함수

In [0]:
import threading
import time

def start_stream_for_domain(bronze_path: str):
    """
    단일 도메인에 대해:
      1) 규칙 없으면 01_rules_generator 자동 실행
      2) Structured Streaming 쿼리 시작
      3) queries 리스트에 추가
    """
    source_name = bronze_path.rstrip("/").split("/")[-1]
 
    # 규칙 탐색
    matched_key = None
    for key in r.keys("gx_rules:*"):
    # Redis 키 자체가 source_name 기반이므로 키 이름으로 직접 비교
        key_source = key.replace("gx_rules:", "")
        if key_source == source_name.lower():
            matched_key = key
            break
 
    # 규칙 없으면 자동 생성
    if matched_key is None:
        print(f"  [INFO] {source_name} 규칙 없음 → 01_rules_generator 실행")
        dbutils.notebook.run(
            "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/notebooks/01_rules_generator",
            timeout_seconds=300,
            arguments={"bronze_path": bronze_path}
        )
        print(f"  [OK] {source_name} 규칙 생성 완료")
    else:
        print(f"  [OK] {source_name} 기존 규칙 사용: {matched_key}")
 
    # 스트리밍 쿼리 시작
    checkpoint_path = f"{BASE_PATH}/_checkpoints/{source_name}"
    spark.conf.set("spark.sql.streaming.stopTimeout", "60000")
    df_stream = (
        spark.readStream
        .format("delta")
        .option("maxFilesPerTrigger", 1)
        .option("maxBytesPerTrigger", "50mb")
        .load(bronze_path)
    )
    query = (
        df_stream
        .writeStream
        .foreachBatch(
            lambda df, epoch_id, sn=source_name: process_batch(df, epoch_id, sn)
        )
        .option("checkpointLocation", checkpoint_path)
        .trigger(processingTime="30 seconds")
        .start()
    )
    queries.append((source_name, query))
    print(f"  [OK] {source_name} 스트리밍 시작")

def watch_new_domains(interval: int = 300):
    """5분마다 Bronze 새 도메인 감지 → 스트리밍 자동 추가"""
    while True:
        time.sleep(interval)
        try:
            existing = {
                bronze_path.rstrip("/").split("/")[-1]
                for bronze_path in domains
            }
            all_paths = [
                f"abfss://bronze@datacopsadls.dfs.core.windows.net/{item.name.rstrip('/')}"
                for item in storage_client.list_directory_contents("bronze")
                if not item.name.startswith("_")
            ]
            for bronze_path in all_paths:
                source_name = bronze_path.rstrip("/").split("/")[-1]

                # ── 이미 실행 중인 도메인 스킵 ──────────────
                if source_name in existing:
                    continue

                # ── 이미 실행 중인 쿼리 이름 확인 ────────────
                running = {q.name for q in spark.streams.active}
                if source_name in running:
                    continue

                print(f"\n[NEW] 새 도메인 감지: {source_name} → 스트리밍 추가")
                domains.append(bronze_path)
                existing.add(source_name)
                start_stream_for_domain(bronze_path)

        except Exception as e:
            print(f"  [WARN] 도메인 감지 오류: {e}")

In [0]:
CHECKPOINT_PATH = f"{BASE_PATH}/_checkpoints/bronze2silver"



# ── Structured Streaming 시작 ─────────────────────────────
queries = []

for bronze_path in domains:
    source_name = bronze_path.rstrip("/").split("/")[-1]
    print(f"\n[START] {source_name} 스트리밍 준비 중...")

    # 규칙 없으면 01 자동 실행
    matched_key = None
    for key in r.keys("gx_rules:*"):
    # Redis 키 자체가 source_name 기반이므로 키 이름으로 직접 비교
        key_source = key.replace("gx_rules:", "")
        if key_source == source_name.lower():
            matched_key = key
            break

    if matched_key is None:
        print(f"  [INFO] {source_name} 규칙 없음 → 01_rules_generator 실행")
        dbutils.notebook.run(
            "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/notebooks/01_rules_generator",
            timeout_seconds=300,
            arguments={"bronze_path": bronze_path}
        )
        print(f"  [OK] {source_name} 규칙 생성 완료")
    else:
        print(f"  [OK] {source_name} 기존 규칙 사용: {matched_key}")

    # 스트림 시작
    checkpoint_path = f"{BASE_PATH}/_checkpoints/{source_name}"
    df_stream = (
        spark.readStream
        .format("delta")
        .option("maxFilesPerTrigger", 1)
        .option("maxBytesPerTrigger", "50mb")
        .load(bronze_path)
    )

    query = (
        df_stream
        .writeStream
        .foreachBatch(
            lambda df, epoch_id, sn=source_name: process_batch(df, epoch_id, sn)
        )
        .option("checkpointLocation", checkpoint_path)
        .trigger(processingTime="30 seconds")
        .start()
    )
    queries.append((source_name, query))
    print(f"  [OK] {source_name} 스트리밍 시작")

print(f"\n[실행 중] 전체 {len(queries)}개 도메인")
for name, _ in queries:
    print(f"  - {name}")

watcher = threading.Thread(target=watch_new_domains, daemon=True)
watcher.start()

spark.streams.awaitAnyTermination()